# Week 4 · The quality lab: noise, signals and the vital few

**Sentence of the day:** *Every process varies. A control chart tells you when a change is a signal and not noise; a Pareto chart tells you which cause to fix first.*

**What you will build today:** the four tools from this morning's case in Python – a Pareto chart, an x̄ and R chart, a p-chart and a capability check – first on Backhaus Kaya's data, then on **your own restaurant from the MyLab Quality simulation** (Part 3). At the end your team has the two charts a quality manager shows first: *which cause* and *when it changed*.

**How this notebook works**

- Google Colab is a notebook: a document where text and runnable code sit side by side. You run a code cell with the ▶ button on its left, or with **Shift + Enter**.
- Run the cells **from top to bottom**. A cell can only use what the cells above it have created.
- **Part 1** is complete: we run it together. In **Part 2** and **Part 3** your team completes the code. Cells marked **✏️ YOUR TURN** contain blanks `___` (three underscores): replace each blank with your code.
- Every step ends with a **check cell**. It prints ✅ when your result matches the hand calculation, and ❌ with a hint when it does not. As long as a blank is left, the check shows ❌.
- You may use an AI assistant. Check its numbers with the check cells and by hand (Part 4, E3, shows how).

| Part | What | Time |
|---|---|---|
| 0 | Setup (run first) | 1 min |
| 1 | Worked example: Kaya's ledger and night log – Pareto, x̄ and R (complete) | 10 min, together |
| 2 | Practice: cost of quality, the p-chart, capability | 15 min · split the team: P1 and P2 + P3 |
| 3 | Your restaurant: the MyLab year as a Pareto chart, a cost-of-quality table and a p-chart | 15 min |
| 4 | Optional extensions for quick teams | if time |

## Part 0 · Setup

This cell loads four **libraries** (toolboxes other people have written): `pandas` for tables, `numpy` for lists of numbers, `matplotlib` for charts, `math` for square roots. All are installed in Colab. It also sets our colours and defines `check`, the helper that prints ✅ or ❌. Run it once.

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

GREY, RED, DARK, GREEN, LIGHT = "#a6a6a6", "#c1002a", "#1a1a1a", "#2e7d4f", "#f2f2f2"
A2, D4, D3 = 0.577, 2.115, 0.0      # control-chart factors for samples of n = 5 (Heizer, Supplement 6, Table S6.1)
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

# Colab normally keeps old outputs under the name ___ . We switch that off, so a blank ___ that
# you have not filled in stops the cell with a clear message instead of using an old output.
try:
    get_ipython().displayhook.cache_size = 0
    get_ipython().user_ns.pop("___", None)
except NameError:
    pass


def _same(got, expected, tol):
    if isinstance(expected, (list, tuple)):
        return len(list(got)) == len(expected) and all(_same(g, e, tol) for g, e in zip(list(got), expected))
    if isinstance(expected, str):
        return str(got) == expected
    return abs(float(got) - float(expected)) <= tol


def _show(x):
    if isinstance(x, (list, tuple)):
        return "[" + ", ".join(_show(i) for i in x) + "]"
    if isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_)):
        return f"{x:,.4f}".rstrip("0").rstrip(".") if float(x) != int(x) else f"{int(x):,}"
    return str(x)


def check(label, got, expected, tol=0.01, hint="", accept=()):
    """Compare your result with the hand calculation and print ✅ or ❌."""
    try:
        ok = any(_same(got, e, tol) for e in [expected, *accept])
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + f"{label}: you have {_show(got)}, expected {_show(expected)}"
          + ("" if ok else f"   Hint: {hint}"))


def not_yet(error):
    if "___" in str(error) or isinstance(error, NameError):
        print("❌ Not done yet: the cell above has not run to its end. Replace every ___ there, run it, then run this check again.")
    else:
        print("❌ Not working yet:", error)


print("Setup done.")

## Part 1 · Worked example: Kaya's ledger and night log

Frau Lindner's ledger has 130 complaints in four weeks. Lena weighed five rolls a night for twenty nights. Here is what she did, in Python.

### 1a · The Pareto chart: which cause first?

**Share of a cause = count ÷ total × 100 %.** Sort the causes from the biggest down and add the shares: the **cumulative share**. The first bars that reach about 70–80 % are the **vital few**.

We store the ledger as a **DataFrame** (a table with named columns). `sort_values` sorts it, `cumsum` adds up the shares from the top.

In [ ]:
ledger = pd.DataFrame([
    ["Underweight (below 56 g)", 52],
    ["Burnt or too dark", 34],
    ["Late delivery", 17],
    ["Torn bags", 11],
    ["Pale or soft crust", 9],
    ["Other", 7]
], columns=["cause", "count"])


def pareto(df, cause="cause", count="count"):
    """Sort by count (biggest first) and add share and cumulative share in %."""
    out = df.sort_values(count, ascending=False).reset_index(drop=True).copy()
    out["share_pct"] = out[count] / out[count].sum() * 100
    out["cum_pct"] = out["share_pct"].cumsum()
    return out


pareto_kaya = pareto(ledger)
pareto_kaya.round(1)

In [ ]:
# CHECK 1a
try:
    check("total complaints", int(pareto_kaya["count"].sum()), 130, tol=0)
    check("shares % (biggest first)", list(pareto_kaya["share_pct"].round(1)), [40.0, 26.2, 13.1, 8.5, 6.9, 5.4])
    check("cumulative %", list(pareto_kaya["cum_pct"].round(1)), [40.0, 66.2, 79.2, 87.7, 94.6, 100.0])
    check("top two causes together (%)", round(float(pareto_kaya["cum_pct"].iloc[1]), 1), 66.2)
    check("the vital few", list(pareto_kaya["cause"].iloc[:2]), ["Underweight (below 56 g)", "Burnt or too dark"])
except Exception as e:
    not_yet(e)

### 1b · Chart 1: the Pareto chart

Bars = counts, biggest first; the line = cumulative share on the right axis. The two red bars are the vital few.

In [ ]:
def draw_pareto(pt, title, highlight=2, cause="cause", count="count"):
    fig, ax = plt.subplots(figsize=(8, 4))
    colours = [RED if i < highlight else GREY for i in range(len(pt))]
    ax.bar(pt[cause], pt[count], color=colours)
    for i, v in enumerate(pt[count]):
        ax.text(i, v + pt[count].max() * 0.01, str(v), ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("complaints")
    ax.set_xticks(range(len(pt)))
    ax.set_xticklabels(pt[cause], rotation=20, ha="right")
    ax2 = ax.twinx()                                   # a second y-axis for the line, in %
    ax2.plot(range(len(pt)), pt["cum_pct"], "o-", color=DARK)
    ax2.set_ylim(0, 105); ax2.set_ylabel("cumulative %"); ax2.spines["right"].set_visible(True)
    for i, v in enumerate(pt["cum_pct"]):
        ax2.text(i, v + 3, f"{v:.0f} %", ha="center", fontsize=8, color=DARK)
    ax.set_title(title)
    plt.tight_layout(); plt.show()


draw_pareto(pareto_kaya, "Backhaus Kaya: FrischMarkt complaints, 4 weeks – the vital few are two causes")

### 1c · The x̄ and R chart: noise or signal?

Every process varies. The question is never *whether* the weight moved, but whether it moved **more than the noise**. A control chart draws the noise band from the process's own data:

- **Sample mean x̄** = sum of the five weights ÷ 5; **range R** = largest − smallest.
- **Grand mean x̿** = mean of the twenty x̄; **average range R̄** = mean of the twenty R.
- **x̄-chart:** UCL = x̿ + A₂ × R̄, LCL = x̿ − A₂ × R̄. **R-chart:** UCL = D₄ × R̄, LCL = D₃ × R̄ (for n = 5: A₂ = 0.577, D₄ = 2.115, D₃ = 0.0).
- A point outside the limits is a **signal** (an assignable cause: find it). Everything inside is **noise** (natural variation: leave it alone).

Lena's twenty nights, five rolls each, in grams. Each inner list is one night, in the order the rolls came off the rack.

In [ ]:
weights = np.array([
    [60.6, 61.1, 59.5, 60.0, 59.4],
    [59.4, 57.8, 59.3, 60.6, 56.4],
    [59.1, 58.3, 56.4, 59.3, 58.9],
    [58.2, 59.5, 58.4, 59.4, 57.9],
    [59.0, 61.9, 59.1, 59.4, 59.8],
    [57.4, 60.1, 58.7, 58.6, 61.0],
    [58.5, 58.5, 58.7, 60.7, 57.9],
    [60.3, 58.5, 56.3, 57.9, 60.4],
    [59.0, 57.9, 58.4, 58.3, 60.4],
    [58.5, 58.5, 62.0, 60.4, 59.5],
    [60.9, 57.6, 58.9, 60.0, 60.0],
    [57.8, 58.7, 59.1, 62.4, 59.7],
    [59.9, 57.2, 59.3, 60.9, 60.1],
    [56.2, 55.9, 55.3, 57.0, 53.4],
    [60.2, 58.0, 59.8, 58.2, 61.2],
    [59.1, 58.2, 59.9, 60.1, 60.0],
    [57.6, 59.3, 59.3, 59.1, 60.6],
    [59.9, 58.0, 59.1, 60.0, 59.4],
    [61.2, 60.0, 59.5, 59.6, 59.3],
    [60.1, 57.8, 58.9, 59.0, 57.1]
])


def xbar_r_limits(samples, A2=A2, D4=D4, D3=D3):
    """x̄, R per sample; grand mean, average range and the two charts' limits."""
    samples = np.asarray(samples, dtype=float)
    xbar = samples.mean(axis=1)
    R = samples.max(axis=1) - samples.min(axis=1)
    xbarbar, rbar = xbar.mean(), R.mean()
    lim = dict(xbarbar=xbarbar, rbar=rbar, ucl_x=xbarbar + A2 * rbar, lcl_x=xbarbar - A2 * rbar,
               ucl_r=D4 * rbar, lcl_r=D3 * rbar)
    lim["out_x"] = [i + 1 for i, v in enumerate(xbar) if v > lim["ucl_x"] or v < lim["lcl_x"]]
    lim["out_r"] = [i + 1 for i, v in enumerate(R) if v > lim["ucl_r"] or v < lim["lcl_r"]]
    return xbar, R, lim


xbar, R, lim = xbar_r_limits(weights)
night_log = pd.DataFrame({"night": range(1, 21), "xbar": xbar.round(2), "R": R.round(1)})
print(f"x̿ = {lim['xbarbar']:.2f} g   R̄ = {lim['rbar']:.2f} g")
print(f"x̄-chart: UCL = {lim['ucl_x']:.2f}   LCL = {lim['lcl_x']:.2f}   ·   R-chart: UCL = {lim['ucl_r']:.2f}   LCL = {lim['lcl_r']:.2f}")
print("nights out of control – x̄:", lim["out_x"], "  R:", lim["out_r"])
night_log.T

In [ ]:
# CHECK 1c
try:
    check("grand mean x̿ (g)", round(lim["xbarbar"], 2), 59.05)
    check("average range R̄ (g)", round(lim["rbar"], 2), 3.0)
    check("x̄-chart UCL", round(lim["ucl_x"], 2), 60.78, hint="x̿ + 0.577 × R̄")
    check("x̄-chart LCL", round(lim["lcl_x"], 2), 57.32)
    check("R-chart UCL", round(lim["ucl_r"], 2), 6.35, hint="2.115 × R̄")
    check("nights out of control (x̄)", lim["out_x"], [14], tol=0)
    check("nights out of control (R)", lim["out_r"], [], tol=0)
except Exception as e:
    not_yet(e)

### 1d · Chart 2: the x̄ and R chart

Night 14 (x̄ = 55.56 g) is below the LCL: the **new flour batch** – stiffer dough, the divider cuts by volume, so the pieces were lighter. The R-chart is quiet: the spread did not change, the centre jumped. That is the signature of a shift, not of a sloppy night. Without night 14 the other nineteen nights average 59.23 g: the divider is set low **every** night – that is the process, not a signal.

In [ ]:
def draw_xbar_r(xbar, R, lim, title, unit="g"):
    n = len(xbar); x = np.arange(1, n + 1)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True, gridspec_kw=dict(height_ratios=[3, 2]))
    for ax, vals, centre, ucl, lcl, lab in ((a1, xbar, lim["xbarbar"], lim["ucl_x"], lim["lcl_x"], f"x̄ ({unit})"),
                                           (a2, R, lim["rbar"], lim["ucl_r"], lim["lcl_r"], f"R ({unit})")):
        out = [(v > ucl) or (v < lcl) for v in vals]
        ax.plot(x, vals, "-", color=DARK, lw=1)
        ax.scatter(x[[not o for o in out]], np.array(vals)[[not o for o in out]], color=DARK, zorder=3)
        ax.scatter(x[out], np.array(vals)[out], color=RED, s=90, zorder=4, label="signal")
        ax.axhline(centre, color=DARK, lw=1.2); ax.axhline(ucl, color=RED, ls="--"); ax.axhline(lcl, color=RED, ls="--")
        ax.text(n + 0.4, ucl, f"UCL {ucl:.2f}", va="center", color=RED, fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
        ax.text(n + 0.4, lcl, f"LCL {lcl:.2f}", va="center", color=RED, fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
        ax.text(n + 0.4, centre, f"{centre:.2f}", va="center", fontsize=8, bbox=dict(fc="white", ec="none", pad=1))
        ax.set_ylabel(lab); ax.set_xlim(0.5, n + 2.5)
    a1.set_title(title); a2.set_xlabel("sample (night)"); a2.set_xticks(x)
    plt.tight_layout(); plt.show()


draw_xbar_r(xbar, R, lim, "Backhaus Kaya: roll weights, 20 nights × 5 rolls – night 14 is a signal, the rest is noise")

## Part 2 · Practice: cost of quality, the p-chart, capability

Three calculations from the case, each with blanks `___`. Split the team: one pair takes P1, the other P2 and P3.

### P1 ✏️ Cost of quality

**Cost of quality = prevention + appraisal + internal failure + external failure** (€ per year). Exhibit B, today:

- prevention: nothing;
- appraisal: Jonas weighs five rolls by eye, 3 minutes a night, 300 nights, 20 € per hour;
- internal failure: 1.5 % of 540000 rolls are thrown away at the bakery, 0.1 € each;
- external failure: 130 complaints in 4 weeks → per year × 50 weeks, 5 € credit each.

Lena's proposal: prevention = calibration contract 400 € + 12 hours of training a year + a logging scale 250 €; appraisal = 10 minutes a night; internal failure halves; 70 % of the *underweight* and *burnt* complaints disappear (the other categories stay).

In [ ]:
WAGE, NIGHTS, ROLLS_YEAR, REVENUE = 20, 300, 540000, 172800

today = {
    "prevention":       0,
    "appraisal":        ___,                               # minutes ÷ 60 × wage × nights
    "internal_failure": ___,                               # scrap share × rolls × cost per roll
    "external_failure": ___,                               # complaints per year × credit
}
top2_share = pareto_kaya["share_pct"].iloc[:2].sum() / 100   # the underweight + burnt share, as a fraction
proposal = {
    "prevention":       400 + 12 * WAGE + 250,
    "appraisal":        ___,
    "internal_failure": today["internal_failure"] / 2,
    "external_failure": ___,                               # today × (1 − top2_share × 0.70)
}


def cost_of_quality(d):
    """Total and share of revenue of a cost-of-quality dictionary."""
    total = sum(d.values())
    return total, total / REVENUE * 100


coq = pd.DataFrame({"today": today, "proposal": proposal})
coq.loc["TOTAL"] = [cost_of_quality(today)[0], cost_of_quality(proposal)[0]]
coq.loc["% of revenue"] = [cost_of_quality(today)[1], cost_of_quality(proposal)[1]]
coq.round(1)

In [ ]:
# CHECK P1
try:
    check("appraisal today (€)", round(today["appraisal"]), 300.0, tol=1, hint="3 ÷ 60 × 20 × 300")
    check("internal failure today (€)", round(today["internal_failure"]), 810.0, tol=1, hint="0.015 × 540,000 × 0.10")
    check("external failure today (€)", round(today["external_failure"]), 8125.0, tol=1, hint="130 ÷ 4 × 50 × 5")
    check("total today (€)", round(cost_of_quality(today)[0]), 9235.0, tol=1)
    check("today as % of revenue", round(cost_of_quality(today)[1], 1), 5.3)
    check("external failure, proposal (€)", round(proposal["external_failure"], 1), 4362.5, tol=0.6, hint="8,125 × (1 − 0.6615 × 0.7)")
    check("total proposal (€)", round(cost_of_quality(proposal)[0], 1), 6657.5, tol=0.6)
    check("saved per year (€)", round(cost_of_quality(today)[0] - cost_of_quality(proposal)[0], 1), 2577.5, tol=0.6)
except Exception as e:
    not_yet(e)

**Read the table:** more than nine tenths of today's cost of quality is *external failure* – the category nobody at Kaya ever wrote down. Lena spends on prevention and appraisal and the total falls. That is Table 6.1 in your book, in euros.

### P2 ✏️ The p-chart: burnt rolls

Burnt or not is an **attribute** (yes/no), so the chart tracks a **proportion**: **p̄ = total defectives ÷ total inspected; σ_p = √( p̄ (1 − p̄) ÷ n ); UCL = p̄ + 3 σ_p; LCL = p̄ − 3 σ_p, never below 0.** Exhibit D: twenty nights, 200 rolls inspected each night.

In [ ]:
burnt = np.array([4, 4, 3, 6, 4, 3, 2, 9, 18, 7, 8, 3, 4, 11, 4, 5, 5, 6, 5, 7])
N_INSPECTED = 200


def p_chart_limits(defectives, n, z=3):
    """p per sample, p̄, σ_p and the limits of a p-chart (LCL floored at 0)."""
    defectives = np.asarray(defectives, dtype=float)
    p = defectives / n
    pbar = ___                                   # total defectives ÷ total inspected
    sigma_p = ___                                # math.sqrt(pbar * (1 - pbar) / n)
    ucl = ___
    lcl = max(0.0, ___)
    out = [i + 1 for i, v in enumerate(p) if v > ucl or v < lcl]
    return p, dict(pbar=pbar, sigma_p=sigma_p, ucl=ucl, lcl=lcl, out=out)


p_burnt, plim = p_chart_limits(burnt, N_INSPECTED)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in plim.items()})

In [ ]:
# CHECK P2
try:
    check("p̄", round(plim["pbar"], 4), 0.0295, tol=0.0001, hint="118 ÷ (20 × 200)")
    check("σ_p", round(plim["sigma_p"], 4), 0.012, tol=0.0001)
    check("UCL", round(plim["ucl"], 4), 0.0655, tol=0.0002, hint="p̄ + 3 × σ_p")
    check("LCL", round(plim["lcl"], 4), 0.0, tol=0.0001, hint="the formula gives a negative number: a proportion cannot be below 0")
    check("night(s) out of control", plim["out"], [9], tol=0)
except Exception as e:
    not_yet(e)

In [ ]:
def draw_p_chart(p, plim, title, xlabel="sample (night)"):
    n = len(p); x = np.arange(1, n + 1); p = np.asarray(p)
    out = [(v > plim["ucl"]) or (v < plim["lcl"]) for v in p]
    fig, ax = plt.subplots(figsize=(9, 3.8))
    ax.plot(x, p, "-", color=DARK, lw=1); ax.scatter(x, p, color=[RED if o else DARK for o in out], s=[90 if o else 36 for o in out], zorder=3)
    for y, c, lab in ((plim["pbar"], DARK, "p̄"), (plim["ucl"], RED, "UCL"), (plim["lcl"], RED, "LCL")):
        ax.axhline(y, color=c, ls="-" if c == DARK else "--"); ax.text(n + 0.4, y, f"{lab} {y:.3f}", va="center", fontsize=8, color=c, bbox=dict(fc="white", ec="none", pad=1))
    ax.set_xlim(0.5, n + 2.5); ax.set_xticks(x); ax.set_ylabel("proportion defective"); ax.set_xlabel(xlabel); ax.set_title(title)
    plt.tight_layout(); plt.show()


draw_p_chart(p_burnt, plim, "Backhaus Kaya: burnt rolls in 200, 20 nights – night 9 is the thermostat")

**Two different questions:** the chart says night 9 is a signal and the other nights are noise. Annex Q says at most 2 % burnt. An ordinary night runs at about 3 % – *in control* and *not good enough* at the same time. The chart judges stability; the spec judges the customer.

### P3 ✏️ Capability: can the line meet 56–64 g at all?

**C_p = (USL − LSL) ÷ (6 σ)** asks whether the spread would fit the window. **C_pk = the smaller of (USL − x̄) ÷ (3 σ) and (x̄ − LSL) ÷ (3 σ)** also asks where the process sits. Capable: at least 1.00; good practice 1.33. Lena's σ = 1.2 g; x̄ = the grand mean from Part 1.

In [ ]:
LSL, USL, TARGET, SIGMA = 56, 64, 60, 1.2


def capability(mean, sigma, lsl=LSL, usl=USL):
    cp = ___
    cpk = min(___, ___)
    return cp, cpk


cp, cpk = capability(lim["xbarbar"], SIGMA)
cp_c, cpk_c = capability(___, SIGMA)          # if Tarek re-sets the divider to the target
print(f"today: Cp = {cp:.2f}, Cpk = {cpk:.2f}   ·   centred at {TARGET} g: Cpk = {cpk_c:.2f}")

In [ ]:
# CHECK P3
try:
    check("Cp", round(cp, 2), 1.11, hint="(64 − 56) ÷ (6 × 1.2)")
    check("Cpk today", round(cpk, 2), 0.85, hint="the smaller side: (x̄ − 56) ÷ (3 × 1.2)")
    check("Cpk if centred at 60 g", round(cpk_c, 2), 1.11)
    check("σ needed for Cp = 1.33", round((USL - LSL) / (6 * 1.33), 2), 1.0, tol=0.011)
except Exception as e:
    not_yet(e)

**Order of work:** centring is free (re-set the divider on Monday); reducing the spread costs money (calibration, a method, a scale). Do the free thing first, then measure again.

## Part 3 ✏️ Your restaurant: the MyLab year in three charts

This afternoon your team ran a restaurant for one simulated year in the **MyLab Quality simulation**: you read customer and staff feedback, chose investments, and were judged on profit and reviews. Your **tally sheet** holds the numbers. Type them in below – the three functions from Parts 1–2 do the rest.

The example values are placeholders so that the cells run before you type: **replace every number and name with your own.** If the simulation's feedback labels differ from the placeholders, use the simulation's labels.

### 3a · The feedback as a Pareto chart
Count every piece of feedback you received by category (your tally sheet, box A).

In [ ]:
feedback = pd.DataFrame([
    ["Slow service",      ___],      # TODO: your counts from the tally sheet
    ["Wrong order",       ___],
    ["Cold food",         ___],
    ["Staff attitude",    ___],
    ["Cleanliness",       ___],
    ["Price / value",     ___],
], columns=["cause", "count"])

pareto_rest = pareto(feedback)
draw_pareto(pareto_rest, "Our restaurant: feedback by category – which two causes did we attack?")
pareto_rest.round(1)

In [ ]:
# CHECK 3a (structure only: your numbers are yours)
try:
    check("shares add up to 100 %", round(float(pareto_rest["share_pct"].sum()), 1), 100.0)
    check("sorted biggest first", bool((pareto_rest["count"].diff().dropna() <= 0).all()), "True")
    print("Top two causes:", list(pareto_rest["cause"].iloc[:2]), "=", round(float(pareto_rest["cum_pct"].iloc[1]), 1), "% of the feedback")
except Exception as e:
    not_yet(e)

### 3b · What you spent, and what failure cost you
List your investments (tally sheet, box B) and sort each into **prevention** (stops the fault: training, better equipment, a new method), **appraisal** (checks: inspection, mystery guests, surveys) or **failure** (what poor quality cost you: refunds, comps, lost customers – the simulation's own numbers if it shows them, otherwise your estimate). Then look at the three shares: did your money go where the Pareto chart pointed?

In [ ]:
investments = pd.DataFrame([
    # name,                        cost,   type ("prevention" / "appraisal" / "failure"),  the cause it addresses
    ["Staff training",             ___,    "prevention",  "Slow service"],      # TODO: your investments
    ["Kitchen equipment",          ___,    "prevention",  "Cold food"],
    ["Mystery-guest programme",    ___,    "appraisal",   "Staff attitude"],
    ["Refunds and free meals",     ___,    "failure",     "all"],
], columns=["item", "cost", "type", "addresses"])

by_type = investments.groupby("type")["cost"].sum().reindex(["prevention", "appraisal", "failure"]).fillna(0)
by_type_pct = (by_type / by_type.sum() * 100).round(1)
pd.DataFrame({"€": by_type, "%": by_type_pct})

In [ ]:
# CHECK 3b (structure only)
try:
    check("every investment has a type", bool(investments["type"].isin(["prevention", "appraisal", "failure"]).all()), "True")
    check("percentages add up", round(float(by_type_pct.sum()), 0), 100.0, tol=0.3)
    aimed = investments[investments["type"] != "failure"]["addresses"].isin(pareto_rest["cause"].iloc[:2]).sum()
    print(f"{aimed} of your prevention/appraisal items address one of your top-two causes.")
except Exception as e:
    not_yet(e)

### 3c · Complaints month by month: a p-chart of your year
From the tally sheet (box C): complaints per month and the number of guests (or orders) per month. If the simulation gives you a review score instead of complaint counts, use the number of *negative* reviews as the defectives and all reviews as n.

In [ ]:
GUESTS_PER_MONTH = ___                         # TODO: guests (or orders, or reviews) per month – one number if constant
complaints = np.array([___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___, ___])   # TODO: 12 months

p_month, plim_rest = p_chart_limits(complaints, GUESTS_PER_MONTH)
draw_p_chart(p_month, plim_rest, "Our restaurant: complaints per guest, 12 months – did our investments show up as a shift?", xlabel="month")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in plim_rest.items()})

In [ ]:
# CHECK 3c (structure only)
try:
    check("twelve months", len(complaints), 12, tol=0)
    check("LCL not below 0", bool(plim_rest["lcl"] >= 0), "True")
    first_half, second_half = p_month[:6].mean(), p_month[6:].mean()
    print(f"first half of the year {first_half:.3f} · second half {second_half:.3f} → "
          + ("the rate fell" if second_half < first_half else "the rate did not fall"))
    print("Months outside the limits:", plim_rest["out"], "– a month below the LCL after an investment is the signal you were hoping for.")
except Exception as e:
    not_yet(e)

### 3d · Your verdict (write it here, three sentences)

1. Which two causes made up most of your feedback, and which of your investments addressed them?
2. Was your money prevention, appraisal or failure – and was the mix what Table 6.1 recommends?
3. Did the p-chart show a *shift* after the investment (points below the old centre line), or just noise? What would you do differently in a second year?

*Your answer:* …

## Part 4 · Optional extensions for quick teams

### E1 · Taguchi: every gram off target costs money
The **quality loss function** says the loss grows with the *square* of the distance from the target: **L = D² × C**, where C is the loss at the spec limit (here 0.05 € per roll at 4 g off – a teaching assumption). For a whole process the expected loss per roll is C ÷ 4² × (σ² + (x̄ − target)²).

In [ ]:
C_AT_LIMIT, D_LIMIT = 0.05, 4
k = C_AT_LIMIT / D_LIMIT ** 2
loss_2g = k * 2 ** 2
exp_loss_roll = k * (SIGMA ** 2 + (lim["xbarbar"] - TARGET) ** 2)
exp_loss_centred = k * SIGMA ** 2
print(f"k = {k:.6f} € per g²  ·  a roll 2 g off: {loss_2g:.4f} €  ·  expected loss per roll today: {exp_loss_roll:.5f} €")
print(f"per year today: {exp_loss_roll * ROLLS_YEAR:,.0f} €   ·   centred at 60 g: {exp_loss_centred * ROLLS_YEAR:,.0f} €")
try:
    check("loss of a roll 2 g off (€)", round(loss_2g, 4), 0.0125, tol=0.0001)
    check("expected loss per year today (€)", round(exp_loss_roll * ROLLS_YEAR), 3956, tol=2)
    check("… if centred (€)", round(exp_loss_centred * ROLLS_YEAR), 2430, tol=2)
except Exception as e:
    not_yet(e)

### E2 · Simulate the line: false alarms and a drift
A control chart is a bet: with 3-sigma limits, a stable process throws a point outside about **3 times in 1,000** samples (a *false alarm*). Below, a stable divider at 60 g with σ = 1.2 g runs 200 nights; from night 101 a drift of −2 g begins. How many false alarms before night 100? How many nights does the chart need to catch the drift? Run it several times – the answer varies, and that is the point.

In [ ]:
rng = np.random.default_rng()            # no fixed seed: every run is a new set of nights
sim = rng.normal(60, 1.2, size=(200, 5))
sim[100:] -= 2.0                          # the drift from night 101
xb_sim, R_sim, lim_sim = xbar_r_limits(sim[:100])        # limits from the stable first 100 nights only
xb_all = sim.mean(axis=1)
false_alarms = [i + 1 for i, v in enumerate(xb_all[:100]) if v > lim_sim["ucl_x"] or v < lim_sim["lcl_x"]]
caught = next((i + 1 for i, v in enumerate(xb_all[100:]) if v < lim_sim["lcl_x"]), None)
print("false alarms in 100 stable nights:", false_alarms)
print("the drift is caught after", caught, "night(s)" if caught else "– not caught in 100 nights")
draw_xbar_r(xb_all, sim.max(axis=1) - sim.min(axis=1), lim_sim, "Simulated line: stable, then a −2 g drift from night 101")

**Think:** a 2 g drift is caught within a few nights; a 0.5 g drift would hide for weeks. Which other chart or rule would catch a small drift sooner? (Your book: run rules – several points in a row on one side of the centre line.)

### E3 · Check an AI's answer
Paste this prompt into an AI assistant of your choice, then compare with your Part 2 result:

> *I am a first-year engineering student. A bakery inspects 200 rolls a night for 20 nights and finds 118 burnt rolls in total. Compute p̄, σ_p and the 3-sigma control limits of a p-chart, showing every step and unit. State every assumption.*

In [ ]:
ai_ucl = ___            # TODO: the AI's UCL, as a proportion (e.g. 0.0655)
try:
    check("AI's UCL vs ours", round(ai_ucl, 4), round(plim["ucl"], 4), tol=0.0003, hint="did the AI use n = 200 (one night) or n = 4,000 (all nights)? The chart needs the sample size, 200")
except Exception as e:
    not_yet(e)

**AI reflection** (two sentences): Was the AI right? The most common AI error here is σ_p with the *total* count instead of the *sample size* – which of your checks would have caught it?

## What to remember

- **Pareto:** share = count ÷ total; sort, add up, fix the vital few first.
- **Cost of quality** = prevention + appraisal + internal failure + external failure; failure is the biggest and the least visible category.
- **x̄-chart:** x̿ ± A₂ R̄; **R-chart:** D₄ R̄ and D₃ R̄; a point outside = a signal, find the cause; inside = noise, leave it alone.
- **p-chart:** p̄ ± 3 √(p̄(1 − p̄)/n), never below 0.
- **C_p** = window ÷ 6σ, **C_pk** = the nearer side ÷ 3σ; centre first (free), then the spread.
- *In control* and *capable* are two different questions.
- Quality is a footprint cut: a roll that comes back (56.5 g CO₂e) or gives flour away is waste twice.

**Sources.** Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed., Global ed.). Pearson. Chapter 6 (cost of quality, Table 6.1; the seven tools; Taguchi) and Supplement 6 (x̄, R and p charts, Table S6.1; process capability). · Pearson MyLab Operations Management, *Quality Management* simulation.

*Transparency note:* Claude (Anthropic) was used to draft and check this material; all figures were computed in Python; all content was reviewed by and is the responsibility of Roman Bartnik.